# From Prompt to Verdict: Building LLM Judges That Actually Work

### A 2-Hour Hands-On Build-Along Bootcamp

Every AI team eventually faces the same wall: your model generates answers, but **how do you know they're any good?** Manual review doesn't scale. BLEU and ROUGE miss the point. And asking an LLM "is this correct?" gives you confident nonsense.

In this bootcamp, you'll build a **production-grade LLM evaluation system from scratch** — line by line, failure by failure, fix by fix.

**46 techniques. 9 blocks. 120 minutes. You code everything.**

---

#### Workshop Roadmap
| Block | Time | What You Build |
|-------|------|---------------|
| 1. Why Evaluation Is Broken | 0–15 min | See naive judges fail with your own eyes |
| 2. Structured Evaluation | 15–35 min | JSON output + rubric + CoT + oath → `evaluate()` |
| 3. Grounding & Critique | 35–55 min | Reference-aware + few-shot + adversarial → `critique()` |
| 4. Comparison Modes | 55–70 min | Pairwise + listwise + bias detection & debiasing |
| 5. Reliability & Consistency | 70–85 min | Self-consistency + confidence thresholds + claim decomposition |
| 6. Multi-Judge Architectures | 85–100 min | Ensemble + specialized + meta-judge + cascading + LLM Council |
| 7. Advanced Techniques | 100–112 min | Dynamic rubrics + hallucination detection + domain adaptation |
| 8. Production & Benchmarking | 112–118 min | Human baselines + monitoring + audit trails |
| 9. Wrap-Up | 118–120 min | Complete class + cheat sheet + resources |

---
## BLOCK 1: WHY EVALUATION IS BROKEN (0–15 min)
*You'll see exactly why naive LLM judges fail — and build the motivation for everything that follows.*

In [ ]:
# Cell 1 — Setup
# Install dependencies (uncomment if needed)
# !pip install litellm numpy scipy pandas

import json
import re
import statistics
import numpy as np
from collections import Counter
from typing import Dict, List, Any, Optional
from litellm import completion

# ── Configure your model ─────────────────────────────────────────────
# LiteLLM supports 100+ providers with a unified API.
# Examples:
#   "ollama/llama3.1:8b"          — local Ollama
#   "gpt-4o-mini"                 — OpenAI
#   "anthropic/claude-3-haiku"    — Anthropic
#   "groq/llama-3.1-8b-instant"  — Groq

MODEL = "ollama/llama3.1:8b"  # ← change this to your model

# Quick connection test
try:
    test = completion(model=MODEL, messages=[{"role": "user", "content": "Say OK"}], max_tokens=5)
    print(f"✅ Connected to {MODEL}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("Check your MODEL string and ensure the provider is running.")


In [ ]:
# Cell 1b — Load test cases we'll use throughout the workshop
with open("../datasets/eval_test_cases.json") as f:
    TEST_CASES = json.load(f)

# We'll use these 3 cases repeatedly to see scores improve as techniques stack
EASY   = TEST_CASES[0]   # Capital of France — correct answer
TRICKY = TEST_CASES[3]   # Coffee & health — sounds right but misses nuance
ADVERSARIAL = TEST_CASES[9]  # Population of Mars — pure hallucination

print("📋 Loaded 12 test cases. Using 3 recurring cases:")
for label, case in [("EASY", EASY), ("TRICKY", TRICKY), ("ADVERSARIAL", ADVERSARIAL)]:
    print(f"\n  [{label}] Q: {case['question']}")
    print(f"           A: {case['answer'][:80]}...")


In [ ]:
# Cell 2 — Traditional Metrics Are Blind
# Let's see why Exact Match and F1 fail for LLM evaluation

def exact_match(reference: str, candidate: str) -> int:
    """Returns 1 if strings are identical, 0 otherwise."""
    return 1 if reference.strip() == candidate.strip() else 0

def token_f1(reference: str, candidate: str) -> float:
    """Token-level F1 score between reference and candidate."""
    ref_tokens = set(reference.lower().split())
    cand_tokens = set(candidate.lower().split())
    common = ref_tokens & cand_tokens
    if not common:
        return 0.0
    precision = len(common) / len(cand_tokens)
    recall = len(common) / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

# Test 1: Correct paraphrase scores ZERO on Exact Match
ref = "The capital of France is Paris."
paraphrase = "Paris serves as France's capital city."
print("Test 1: Correct paraphrase")
print(f"  Reference:  '{ref}'")
print(f"  Candidate:  '{paraphrase}'")
print(f"  Exact Match: {exact_match(ref, paraphrase)}  ← WRONG! It's correct but scores 0")
print(f"  Token F1:    {token_f1(ref, paraphrase):.2f}  ← Low despite being correct")

# Test 2: Hallucination with right keywords scores HIGH
hallucination = "The capital of France is Paris, and it was founded by the Roman Empire in 250 BC as Lutetia, which later became the center of the French Revolution."
print(f"\nTest 2: Hallucinated answer with correct keywords")
print(f"  Reference:  '{ref}'")
print(f"  Candidate:  '{hallucination[:60]}...'")
print(f"  Token F1:    {token_f1(ref, hallucination):.2f}  ← HIGH despite containing hallucinations")

print("\n💡 LESSON: Traditional metrics measure WORDS, not MEANING.")


In [ ]:
# Cell 3 — Naive Binary Judge (Technique #1)
# The simplest possible judge: "Is this correct? Yes/No"

def ask_llm(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Helper: send a prompt to the LLM and return the response text."""
    resp = completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

# Run naive binary judge on all 3 cases
print("🔍 NAIVE BINARY JUDGE: 'Is this answer correct? Yes or No'\n")

for label, case in [("EASY", EASY), ("TRICKY", TRICKY), ("ADVERSARIAL", ADVERSARIAL)]:
    prompt = f"Is this answer correct? Reply only Yes or No.\n\nQuestion: {case['question']}\nAnswer: {case['answer']}"
    verdict = ask_llm(prompt)
    expected = case["expected_quality"]
    print(f"  [{label}] Verdict: {verdict:<6} (expected quality: {expected})")

print("\n💡 LESSON: The LLM says 'Yes' to almost everything — even hallucinations.")
print("   LLMs are sycophantic by default. They agree with confident-sounding text.")


In [ ]:
# Cell 4 — Naive Likert Rating (Technique #2)
# "Rate 1-10" — feels scientific but is it?

print("🔍 NAIVE LIKERT RATING: 'Rate this answer 1-10'\n")
print("Running same prompt 3x on the TRICKY case to check consistency...\n")

scores = []
for run in range(3):
    prompt = f"Rate this answer from 1 to 10 for quality. Reply with ONLY the number.\n\nQuestion: {TRICKY['question']}\nAnswer: {TRICKY['answer']}"
    score = ask_llm(prompt)
    scores.append(score)
    print(f"  Run {run+1}: Score = {score}")

print(f"\n  Scores across runs: {scores}")
print("💡 LESSON: Without criteria, scores are arbitrary and inconsistent.")


In [ ]:
# Cell 5 — Naive Pointwise Scoring (Technique #3)
# Ask the LLM to calculate metrics — watch it hallucinate numbers

predictions = ["positive", "negative", "positive", "positive", "negative", "positive"]
ground_truth = ["positive", "positive", "negative", "positive", "positive", "negative"]

prompt = f"""Calculate the accuracy, precision, and recall for these predictions.

Predictions:  {predictions}
Ground Truth: {ground_truth}

Return the exact numbers."""

print("🔍 NAIVE POINTWISE: 'Calculate accuracy, precision, recall'\n")
result = ask_llm(prompt)
print(f"LLM says:\n{result}")

# Now let's calculate the REAL metrics ourselves
tp = sum(1 for p, g in zip(predictions, ground_truth) if p == "positive" and g == "positive")
tn = sum(1 for p, g in zip(predictions, ground_truth) if p == "negative" and g == "negative")
fp = sum(1 for p, g in zip(predictions, ground_truth) if p == "positive" and g == "negative")
fn = sum(1 for p, g in zip(predictions, ground_truth) if p == "negative" and g == "positive")

real_accuracy = (tp + tn) / len(predictions)
real_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
real_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n📊 REAL metrics (calculated by us):")
print(f"  TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print(f"  Accuracy:  {real_accuracy:.3f}")
print(f"  Precision: {real_precision:.3f}")
print(f"  Recall:    {real_recall:.3f}")
print("\n💡 LESSON: LLMs CANNOT do math reliably. Never let them calculate metrics.")


In [ ]:
# Cell 6 — Failure Modes Summary
# Roadmap: what goes wrong and what we'll fix

print("=" * 70)
print("📋 FAILURE MODES SUMMARY — Your Roadmap for the Rest of This Workshop")
print("=" * 70)

failure_modes = [
    ("Binary Yes/No",       "Sycophantic — says Yes to everything",         "Block 2: Structured evaluation"),
    ("Likert 1-10",         "Arbitrary — no criteria, inconsistent scores", "Block 2: Rubric + CoT"),
    ("Calculate metrics",   "Hallucinates numbers — LLMs can't do math",   "Block 5: Component extraction"),
    ("Single evaluation",   "High variance — different answer each run",   "Block 5: Self-consistency"),
    ("No reference",        "Internal knowledge bias — misses gaps",       "Block 3: Reference-grounded"),
    ("Generous by default", "Misses flaws in confident-sounding text",     "Block 3: Adversarial critique"),
    ("Single model",        "Model-specific blind spots",                   "Block 6: Multi-judge ensemble"),
    ("No human baseline",   "No way to know if judge is actually good",    "Block 8: Benchmarking"),
]

print(f"\n{'Approach':<22} {'Failure Mode':<45} {'Fix'}")
print("-" * 100)
for approach, failure, fix in failure_modes:
    print(f"{approach:<22} {failure:<45} {fix}")

print("\n🚀 Let's fix ALL of these, one by one.")


---
## BLOCK 2: STRUCTURED EVALUATION — THE FOUNDATION (15–35 min)
*The #1 most important upgrade: structure everything. Without structure, nothing else works.*

In [ ]:
# Cell 7 — Structured JSON Output (Technique #6)
# Force the LLM to return parseable JSON instead of free-form text

STRUCTURED_SYSTEM = """You are an expert evaluator. Always respond in this exact JSON format:
{
    "verdict": "good" or "poor" or "mixed",
    "reasoning": "your explanation",
    "confidence": "high" or "medium" or "low"
}
Return ONLY valid JSON. No other text."""

def judge_structured(question: str, answer: str) -> dict:
    """Get a structured JSON evaluation from the LLM."""
    prompt = f"Evaluate this answer.\n\nQuestion: {question}\nAnswer: {answer}"
    raw = ask_llm(prompt, system=STRUCTURED_SYSTEM)
    # Strip markdown code fences if present
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"verdict": "parse_error", "raw": raw}

print("🔧 STRUCTURED JSON OUTPUT\n")
for label, case in [("EASY", EASY), ("TRICKY", TRICKY), ("ADVERSARIAL", ADVERSARIAL)]:
    result = judge_structured(case["question"], case["answer"])
    print(f"  [{label}] {json.dumps(result, indent=2)}\n")

print("💡 LESSON: Structured output = parseable, consistent, machine-readable.")


In [ ]:
# Cell 8 — Explicit Rubric / Criteria (Technique #7)
# Decompose "is it good?" into measurable dimensions

RUBRIC_SYSTEM = """You are an expert evaluator. Evaluate the answer on these specific criteria:

- accuracy (0-3): 0=wrong, 1=partially correct, 2=mostly correct, 3=fully correct
- completeness (0-3): 0=missing key info, 1=surface-level, 2=good coverage, 3=comprehensive
- clarity (0-3): 0=confusing, 1=understandable, 2=clear, 3=excellent

Respond in this exact JSON format:
{
    "accuracy": {"score": 0-3, "justification": "why this score"},
    "completeness": {"score": 0-3, "justification": "why this score"},
    "clarity": {"score": 0-3, "justification": "why this score"},
    "overall_score": "sum of three scores (0-9)",
    "verdict": "good (7-9) / mixed (4-6) / poor (0-3)"
}
Return ONLY valid JSON."""

def judge_rubric(question: str, answer: str) -> dict:
    """Evaluate using explicit rubric criteria."""
    prompt = f"Question: {question}\nAnswer: {answer}"
    raw = ask_llm(prompt, system=RUBRIC_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

print("🔧 RUBRIC-BASED EVALUATION\n")
for label, case in [("EASY", EASY), ("TRICKY", TRICKY), ("ADVERSARIAL", ADVERSARIAL)]:
    result = judge_rubric(case["question"], case["answer"])
    if "parse_error" not in result:
        scores = f"acc={result.get('accuracy',{}).get('score','?')} comp={result.get('completeness',{}).get('score','?')} clar={result.get('clarity',{}).get('score','?')}"
        print(f"  [{label}] {scores} → overall={result.get('overall_score','?')} verdict={result.get('verdict','?')}")
    else:
        print(f"  [{label}] Parse error — raw: {result['raw'][:100]}...")

print("\n💡 LESSON: Explicit criteria let us diagnose WHERE an answer fails, not just IF.")


In [ ]:
# Cell 9 — Decomposed Multi-Criteria Evaluation (Technique #8)
# Expand rubric into a detailed checklist with binary and scaled sub-criteria

DECOMPOSED_SYSTEM = """You are an expert evaluator. Break down the evaluation into a detailed checklist.

Evaluate each sub-criterion independently:
- addresses_main_question: (0 or 1) Does it answer what was actually asked?
- factual_accuracy: (0-3) Are the stated facts correct?
- evidence_provided: (0-2) Does it provide supporting details or examples?
- acknowledges_limitations: (0-2) Does it mention caveats, edge cases, or nuance?
- no_misleading_claims: (0 or 1) Is it free of misleading or false implications?
- appropriate_depth: (0 or 1) Is the level of detail appropriate for the question?

Respond in this exact JSON format:
{
    "addresses_main_question": {"score": 0, "note": "..."},
    "factual_accuracy": {"score": 0, "note": "..."},
    "evidence_provided": {"score": 0, "note": "..."},
    "acknowledges_limitations": {"score": 0, "note": "..."},
    "no_misleading_claims": {"score": 0, "note": "..."},
    "appropriate_depth": {"score": 0, "note": "..."},
    "total": "sum out of 10",
    "verdict": "good (8-10) / mixed (5-7) / poor (0-4)"
}
Return ONLY valid JSON."""

def judge_decomposed(question: str, answer: str) -> dict:
    """Evaluate using decomposed multi-criteria checklist."""
    prompt = f"Question: {question}\nAnswer: {answer}"
    raw = ask_llm(prompt, system=DECOMPOSED_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

print("🔧 DECOMPOSED MULTI-CRITERIA EVALUATION\n")
result = judge_decomposed(TRICKY["question"], TRICKY["answer"])
if "parse_error" not in result:
    for key in ["addresses_main_question", "factual_accuracy", "evidence_provided",
                 "acknowledges_limitations", "no_misleading_claims", "appropriate_depth"]:
        entry = result.get(key, {})
        print(f"  {key}: {entry.get('score', '?')} — {entry.get('note', '?')}")
    print(f"\n  Total: {result.get('total', '?')}  Verdict: {result.get('verdict', '?')}")
else:
    print(f"  Parse error: {result['raw'][:200]}...")

print("\n💡 LESSON: More dimensions = better diagnostic power. You can see EXACTLY what failed.")


In [ ]:
# Cell 10 — Chain-of-Thought Reasoning (Technique #9)
# Force the LLM to explain reasoning BEFORE scoring

COT_SYSTEM = """You are an expert evaluator who thinks step by step.

MANDATORY PROCESS — follow in this exact order:
1. Identify what the question is actually asking
2. Check the answer's factual claims one by one
3. Note any gaps, nuances, or caveats the answer misses
4. Consider whether the answer could be misleading
5. THEN assign scores based on your analysis

Rubric: accuracy (0-3), completeness (0-3), clarity (0-3)

Respond in JSON:
{
    "reasoning_steps": [
        "Step 1: ...",
        "Step 2: ...",
        "Step 3: ...",
        "Step 4: ..."
    ],
    "accuracy": {"score": 0, "justification": "..."},
    "completeness": {"score": 0, "justification": "..."},
    "clarity": {"score": 0, "justification": "..."},
    "overall_score": "sum (0-9)",
    "verdict": "good/mixed/poor",
    "issues_found": ["issue1", "issue2"]
}
Return ONLY valid JSON."""

def judge_cot(question: str, answer: str) -> dict:
    """Evaluate with mandatory chain-of-thought reasoning."""
    prompt = f"Question: {question}\nAnswer: {answer}"
    raw = ask_llm(prompt, system=COT_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

print("🔧 CHAIN-OF-THOUGHT EVALUATION\n")
print("Testing on TRICKY case (coffee & health):\n")
result = judge_cot(TRICKY["question"], TRICKY["answer"])

if "parse_error" not in result:
    print("Reasoning steps:")
    for step in result.get("reasoning_steps", []):
        print(f"  → {step}")
    print(f"\nScores: acc={result.get('accuracy',{}).get('score','?')} "
          f"comp={result.get('completeness',{}).get('score','?')} "
          f"clar={result.get('clarity',{}).get('score','?')}")
    print(f"Issues found: {result.get('issues_found', [])}")
    print(f"Verdict: {result.get('verdict', '?')}")
else:
    print(f"Parse error: {result['raw'][:300]}...")

print("\n💡 LESSON: CoT surfaces problems that direct scoring skips."
      " The LLM catches nuance issues when forced to reason first.")


In [ ]:
# Cell 11 — Oath-Based Consistency (Technique #10)
# Add a behavioral commitment to reduce scoring drift

OATH_SYSTEM = """You are an expert evaluator. Before evaluating, you must take this oath:

OATH: "I promise to:
1. Evaluate based ONLY on the provided criteria and evidence
2. Maintain consistent standards across all evaluations
3. Acknowledge uncertainty when present rather than guessing
4. Not be swayed by confident language or length of answer
5. Find flaws even in well-written responses"

After taking the oath, evaluate using:
- accuracy (0-3), completeness (0-3), clarity (0-3)

Respond in JSON:
{
    "oath_taken": true,
    "accuracy": {"score": 0, "justification": "..."},
    "completeness": {"score": 0, "justification": "..."},
    "clarity": {"score": 0, "justification": "..."},
    "overall_score": "sum (0-9)",
    "verdict": "good/mixed/poor",
    "confidence": "high/medium/low"
}
Return ONLY valid JSON."""

# Compare with vs without oath on the same case
print("🔧 OATH-BASED CONSISTENCY\n")
print("Comparing WITH vs WITHOUT oath on the ADVERSARIAL case (Mars population):\n")

without_oath = judge_rubric(ADVERSARIAL["question"], ADVERSARIAL["answer"])
prompt = f"Question: {ADVERSARIAL['question']}\nAnswer: {ADVERSARIAL['answer']}"
with_oath_raw = ask_llm(prompt, system=OATH_SYSTEM)
cleaned = re.sub(r"```json\s*|\s*```", "", with_oath_raw).strip()
try:
    with_oath = json.loads(cleaned)
except json.JSONDecodeError:
    with_oath = {"parse_error": True, "raw": with_oath_raw}

print(f"  WITHOUT oath: overall={without_oath.get('overall_score','?')} verdict={without_oath.get('verdict','?')}")
if "parse_error" not in with_oath:
    print(f"  WITH oath:    overall={with_oath.get('overall_score','?')} verdict={with_oath.get('verdict','?')}")
else:
    print(f"  WITH oath:    (parse issue) {with_oath.get('raw','')[:150]}...")

print("\n💡 LESSON: Oath-based prompting nudges the LLM toward stricter, more consistent evaluation.")


In [ ]:
# Cell 12 — Build evaluate() function ✅ CHECKPOINT
# Combines: structured output + rubric + CoT + oath into one reusable function

EVALUATE_SYSTEM = """You are an expert evaluator. Take this oath before evaluating:
"I will evaluate based ONLY on criteria and evidence, maintain consistent standards,
acknowledge uncertainty, and not be swayed by confident language or length."

PROCESS:
1. Reason step-by-step about the answer's quality
2. Check factual claims carefully
3. Note gaps, nuances, or misleading elements
4. Score each dimension

Rubric:
- accuracy (0-3): 0=wrong, 1=partial, 2=mostly correct, 3=fully correct
- completeness (0-3): 0=missing key info, 1=surface, 2=good, 3=comprehensive
- clarity (0-3): 0=confusing, 1=ok, 2=clear, 3=excellent

Respond ONLY with valid JSON:
{
    "reasoning": "your step-by-step analysis",
    "accuracy": {"score": 0, "justification": "..."},
    "completeness": {"score": 0, "justification": "..."},
    "clarity": {"score": 0, "justification": "..."},
    "overall_score": 0,
    "verdict": "good/mixed/poor",
    "confidence": "high/medium/low",
    "issues": ["issue1", "issue2"]
}"""

def evaluate(question: str, answer: str, reference: str = None) -> dict:
    """Core evaluation function combining structured output + rubric + CoT + oath.
    Optionally accepts a reference answer for grounded evaluation (Technique #12).
    """
    prompt = f"Question: {question}\nAnswer: {answer}"
    if reference:
        prompt += f"\nReference Answer: {reference}\nCompare the candidate answer against this reference."

    raw = ask_llm(prompt, system=EVALUATE_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

# Test on all 3 recurring cases
print("✅ CHECKPOINT: evaluate() function built!\n")
print("Running evaluate() on all 3 cases:\n")

for label, case in [("EASY", EASY), ("TRICKY", TRICKY), ("ADVERSARIAL", ADVERSARIAL)]:
    result = evaluate(case["question"], case["answer"])
    if "parse_error" not in result:
        print(f"  [{label}] score={result.get('overall_score','?')}/9  "
              f"verdict={result.get('verdict','?')}  "
              f"confidence={result.get('confidence','?')}")
        issues = result.get("issues", [])
        if issues:
            print(f"           issues: {issues}")
    else:
        print(f"  [{label}] parse error")
    print()


---
## BLOCK 3: GROUNDING & CRITIQUE (35–55 min)
*Make your judge accurate by grounding it in reference material and teaching it to find flaws.*

In [ ]:
# Cell 13 — Reference-Grounded Evaluation (Technique #12)
# Compare WITH vs WITHOUT a reference answer

print("🔧 REFERENCE-GROUNDED EVALUATION\n")
print("Same question, same answer — evaluated WITH vs WITHOUT reference:\n")

case = TRICKY  # Coffee & health — sounds right but misses nuance

# Without reference (uses LLM's internal knowledge)
result_no_ref = evaluate(case["question"], case["answer"])

# With reference (grounded in expert knowledge)
result_with_ref = evaluate(case["question"], case["answer"], reference=case["reference_answer"])

print(f"  WITHOUT reference: score={result_no_ref.get('overall_score','?')}/9  verdict={result_no_ref.get('verdict','?')}")
print(f"  WITH reference:    score={result_with_ref.get('overall_score','?')}/9  verdict={result_with_ref.get('verdict','?')}")
print(f"\n  Reference answer (expert): {case['reference_answer'][:150]}...")

print("\n💡 LESSON: Without a reference, the LLM rates based on 'sounds reasonable'.")
print("   With a reference, it can identify specific gaps and inaccuracies.")


In [ ]:
# Cell 14 — Few-Shot Judging (Technique #11)
# Provide example evaluations to anchor the LLM's scoring scale

FEW_SHOT_SYSTEM = """You are an expert evaluator. Here are example evaluations to calibrate your scoring:

EXAMPLE 1 (score 9/9 — excellent):
Question: "What is the capital of France?"
Answer: "Paris is the capital and most populous city of France, situated on the Seine River."
Evaluation: accuracy=3, completeness=3, clarity=3 → 9/9, verdict=good

EXAMPLE 2 (score 3/9 — poor):
Question: "Explain how vaccines work."
Answer: "Vaccines make you immune to diseases."
Evaluation: accuracy=1, completeness=0, clarity=2 → 3/9, verdict=poor
(Oversimplified, missing mechanism, no mention of immune response, antibodies, or memory cells)

EXAMPLE 3 (score 5/9 — mixed):
Question: "Is coffee good for health?"
Answer: "Yes, coffee has many health benefits including antioxidants."
Evaluation: accuracy=2, completeness=1, clarity=2 → 5/9, verdict=mixed
(Partially correct but one-sided, ignores dose-dependency, contraindications, individual variation)

Now evaluate the following using the same standards and rubric (accuracy 0-3, completeness 0-3, clarity 0-3).
Respond ONLY with valid JSON:
{
    "accuracy": {"score": 0, "justification": "..."},
    "completeness": {"score": 0, "justification": "..."},
    "clarity": {"score": 0, "justification": "..."},
    "overall_score": 0,
    "verdict": "good/mixed/poor"
}"""

def judge_few_shot(question: str, answer: str) -> dict:
    """Evaluate with few-shot example anchoring."""
    prompt = f"Question: {question}\nAnswer: {answer}"
    raw = ask_llm(prompt, system=FEW_SHOT_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

print("🔧 FEW-SHOT JUDGING\n")
for label, case in [("EASY", EASY), ("TRICKY", TRICKY), ("ADVERSARIAL", ADVERSARIAL)]:
    result = judge_few_shot(case["question"], case["answer"])
    if "parse_error" not in result:
        print(f"  [{label}] score={result.get('overall_score','?')}/9 verdict={result.get('verdict','?')}")
    else:
        print(f"  [{label}] parse error")

print("\n💡 LESSON: Few-shot examples anchor the scoring scale better than rubric text alone.")


In [ ]:
# Cell 15 — Adversarial Critique (Technique #13)
# Build a critique() function that finds flaws FIRST

CRITIQUE_SYSTEM = """You are a ruthless critic. Your job is to find EVERY flaw in this answer.

MANDATORY ORDER:
1. Find all factual errors or misleading claims
2. Identify missing information that should have been included
3. Note any logical gaps or unsupported assertions
4. Check for overconfidence or false certainty
5. ONLY THEN mention any strengths

Respond ONLY with valid JSON:
{
    "critical_flaws": ["flaw1", "flaw2"],
    "missing_elements": ["missing1", "missing2"],
    "misleading_claims": ["claim1"],
    "strengths": ["strength1"],
    "harsh_score": "0-9 after critical analysis",
    "recommendation": "accept/revise/reject"
}"""

def critique(question: str, answer: str) -> dict:
    """Adversarial evaluation — find all flaws first."""
    prompt = f"Question: {question}\nAnswer: {answer}\n\nFind ALL problems with this answer."
    raw = ask_llm(prompt, system=CRITIQUE_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

# Test on the microservices case — confidently wrong advice
case = TEST_CASES[5]  # Microservices vs monolith
print("🔧 ADVERSARIAL CRITIQUE\n")
print(f"  Q: {case['question']}")
print(f"  A: {case['answer'][:120]}...\n")

# Standard eval
std = evaluate(case["question"], case["answer"])
print(f"  Standard eval:    score={std.get('overall_score','?')}/9 verdict={std.get('verdict','?')}")

# Adversarial critique
crit = critique(case["question"], case["answer"])
if "parse_error" not in crit:
    print(f"  Adversarial score: {crit.get('harsh_score','?')}/9 recommendation={crit.get('recommendation','?')}")
    print(f"  Critical flaws: {crit.get('critical_flaws', [])}")
    print(f"  Missing elements: {crit.get('missing_elements', [])}")
else:
    print(f"  Adversarial: parse error")

print("\n💡 LESSON: LLMs are generous by default. Adversarial mode catches flaws they'd otherwise miss.")


In [ ]:
# Cell 16 — Contrastive Evaluation (Technique #14)
# Ask "what SPECIFICALLY makes one better?" instead of vague preferences

CONTRASTIVE_SYSTEM = """You compare two answers by identifying SPECIFIC, CONCRETE differences.
Do NOT say "A is better." Say exactly WHY and WHERE.

Respond ONLY with valid JSON:
{
    "specific_differences": [
        {"dimension": "...", "answer_a": "...", "answer_b": "...", "better": "A/B/tie"}
    ],
    "overall_winner": "A/B/tie",
    "key_reason": "the single most important difference"
}"""

def contrastive(question: str, answer_a: str, answer_b: str) -> dict:
    """Contrastive evaluation — specific differences, not vague preferences."""
    prompt = f"Question: {question}\nAnswer A: {answer_a}\nAnswer B: {answer_b}"
    raw = ask_llm(prompt, system=CONTRASTIVE_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

with open("../datasets/pairwise_cases.json") as f:
    PAIRWISE_CASES = json.load(f)

pair = PAIRWISE_CASES[2]  # Remote work — subtle difference
print("🔧 CONTRASTIVE EVALUATION\n")
print(f"  Q: {pair['question']}")
print(f"  A: {pair['answer_a'][:80]}...")
print(f"  B: {pair['answer_b'][:80]}...\n")

result = contrastive(pair["question"], pair["answer_a"], pair["answer_b"])
if "parse_error" not in result:
    for diff in result.get("specific_differences", [])[:3]:
        print(f"  [{diff.get('dimension','')}] Better: {diff.get('better','')} — A: {diff.get('answer_a','')[:50]}... B: {diff.get('answer_b','')[:50]}...")
    print(f"\n  Winner: {result.get('overall_winner','')} — {result.get('key_reason','')}")
else:
    print(f"  Parse error: {result['raw'][:200]}...")

print("\n💡 LESSON: Contrastive explanations are more actionable than scores.")


---
## BLOCK 4: COMPARISON MODES & BIAS (55–70 min)
*Pairwise, listwise, and the biases that silently corrupt your judgments.*

In [ ]:
# Cell 18 — Pairwise Comparison (Technique #4)

PAIRWISE_SYSTEM = """You compare two answers to the same question.
Evaluate on: accuracy, completeness, clarity. Pick a winner.

Respond ONLY with valid JSON:
{
    "comparison": {
        "accuracy": {"better": "A/B/tie", "reason": "..."},
        "completeness": {"better": "A/B/tie", "reason": "..."},
        "clarity": {"better": "A/B/tie", "reason": "..."}
    },
    "winner": "A/B/tie",
    "reasoning": "overall justification",
    "confidence": "high/medium/low"
}"""

def compare(question: str, answer_a: str, answer_b: str, label_a: str = "A", label_b: str = "B") -> dict:
    """Pairwise comparison of two answers."""
    prompt = (f"Question: {question}\n"
              f"Answer {label_a}: {answer_a}\n"
              f"Answer {label_b}: {answer_b}")
    raw = ask_llm(prompt, system=PAIRWISE_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

pair = PAIRWISE_CASES[0]  # Clear winner case
print("🔧 PAIRWISE COMPARISON (Technique #4)\n")
print(f"  Q: {pair['question']}")
print(f"  A: {pair['answer_a'][:80]}...")
print(f"  B: {pair['answer_b'][:80]}...\n")

result = compare(pair["question"], pair["answer_a"], pair["answer_b"])
if "parse_error" not in result:
    print(f"  Winner: {result.get('winner','')}  Confidence: {result.get('confidence','')}")
    print(f"  Reason: {result.get('reasoning','')[:120]}")
    print(f"  Expected: {pair['expected_winner']}")
else:
    print(f"  Parse error")

print("\n💡 LESSON: Direct comparison often gives clearer signal than absolute scores.")


In [ ]:
# Cell 19 — Position Bias Detection & Debiasing (Techniques #15, #16)
# LLMs sometimes prefer whichever answer is shown first

print("🔧 POSITION BIAS DETECTION & DEBIASING\n")

bias_count = 0
for i, pair in enumerate(PAIRWISE_CASES[:3]):
    # Order 1: A first, B second
    r1 = compare(pair["question"], pair["answer_a"], pair["answer_b"])
    winner_1 = r1.get("winner", "?") if "parse_error" not in r1 else "?"

    # Order 2: B first, A second (swap labels)
    r2 = compare(pair["question"], pair["answer_b"], pair["answer_a"])
    winner_2 = r2.get("winner", "?") if "parse_error" not in r2 else "?"
    # Map back: if r2 says "A", that's actually original B
    winner_2_mapped = {"A": "B", "B": "A", "tie": "tie"}.get(winner_2, winner_2)

    consistent = winner_1 == winner_2_mapped
    if not consistent:
        bias_count += 1
    print(f"  Pair {i+1}: Order1→{winner_1}  Order2→{winner_2_mapped}  {'✅ Consistent' if consistent else '❌ POSITION BIAS'}")

print(f"\n  Position bias detected in {bias_count}/{min(3, len(PAIRWISE_CASES))} pairs")

def compare_debiased(question: str, answer_a: str, answer_b: str) -> dict:
    """Position-debiased pairwise comparison — runs both orders."""
    r1 = compare(question, answer_a, answer_b)
    r2 = compare(question, answer_b, answer_a)

    w1 = r1.get("winner", "?") if "parse_error" not in r1 else "?"
    w2_raw = r2.get("winner", "?") if "parse_error" not in r2 else "?"
    w2 = {"A": "B", "B": "A", "tie": "tie"}.get(w2_raw, w2_raw)

    if w1 == w2:
        return {"winner": w1, "consistent": True, "confidence": "high"}
    else:
        return {"winner": "tie", "consistent": False, "confidence": "low",
                "note": f"Inconsistent: order1={w1}, order2={w2} — flagged as tie"}

print("\n  Debiased result for Pair 1:")
debiased = compare_debiased(PAIRWISE_CASES[0]["question"], PAIRWISE_CASES[0]["answer_a"], PAIRWISE_CASES[0]["answer_b"])
print(f"  {debiased}")

print("\n💡 LESSON: Always run pairwise comparisons in both orders. Inconsistency = unreliable.")


In [ ]:
# Cell 20 — Length/Verbosity Bias & Sycophancy Bias (Techniques #17-20)

print("🔧 LENGTH BIAS DETECTION (Techniques #17, #18)\n")

# Test: short correct answer vs long wrong-but-detailed answer
short_correct = "The cheetah is the fastest land animal, reaching ~70 mph. However, the peregrine falcon is the fastest animal overall at 240+ mph in a dive."
long_wrong = """The cheetah is widely recognized as the fastest animal on Earth. With its sleek body, 
long legs, and specialized muscles, it can achieve remarkable speeds of up to 70 miles per hour. 
The cheetah's extraordinary speed is a result of millions of years of evolution, developing unique 
adaptations including semi-retractable claws for grip, an enlarged heart and lungs for oxygen delivery, 
and a flexible spine that extends its stride length. These magnificent creatures primarily inhabit 
the African savanna and are unfortunately endangered due to habitat loss and human conflict. 
Their speed makes them apex predators in their ecosystem, capable of catching prey that other 
predators cannot. The cheetah's acceleration from 0 to 60 mph in just 3 seconds rivals that of 
many sports cars, making it one of nature's most impressive athletes."""

q = "What is the fastest animal on Earth?"
r = compare(q, short_correct, long_wrong)
winner = r.get("winner", "?") if "parse_error" not in r else "?"
print(f"  Short (correct + complete) vs Long (partially correct, misses falcon):")
print(f"  Winner: {winner}")
print(f"  Expected: A (short is more accurate and complete)")

print("\n🔧 SYCOPHANCY BIAS (Technique #19)\n")
confident_wrong = "The Great Wall of China is visible from space with the naked eye. This is a well-established fact confirmed by astronauts."
prompt = f"Is this answer correct? Reply only Yes or No.\n\nQuestion: Can you see the Great Wall from space?\nAnswer: {confident_wrong}"
verdict = ask_llm(prompt)
print(f"  Confident-but-wrong claim: '{confident_wrong[:60]}...'")
print(f"  LLM verdict: {verdict}  (should be No — this is a common myth)")

print("\n🔧 ANCHOR BIAS (Technique #20)\n")
anchor_prompt = f"""A previous expert rated this answer 9/10.
Now rate this answer from 1 to 10. Reply with ONLY the number.

Question: {TRICKY['question']}
Answer: {TRICKY['answer']}"""
anchored = ask_llm(anchor_prompt)
unanchored = ask_llm(f"Rate this answer from 1 to 10. Reply with ONLY the number.\n\nQuestion: {TRICKY['question']}\nAnswer: {TRICKY['answer']}")
print(f"  Without anchor: {unanchored}")
print(f"  With anchor (told 'expert gave 9/10'): {anchored}")
print(f"  Difference shows anchor bias pulling score up")

print("\n💡 LESSON: Know your judge's biases — length, sycophancy, anchoring all corrupt results.")


In [ ]:
# Cell 21 — Listwise Ranking (Technique #5)

RANKING_SYSTEM = """You rank multiple answers from best to worst.

Respond ONLY with valid JSON:
{
    "ranking": ["best_label", "second", "third", "worst_label"],
    "reasoning": {"best_label": "why first", "worst_label": "why last"},
    "confidence": "high/medium/low"
}"""

def rank(question: str, answers: List[Dict]) -> dict:
    """Rank multiple answers from best to worst."""
    answers_text = "\n".join(f"Answer {a['label']}: {a['text']}" for a in answers)
    prompt = f"Question: {question}\n\n{answers_text}\n\nRank from best to worst."
    raw = ask_llm(prompt, system=RANKING_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

with open("../datasets/listwise_cases.json") as f:
    LISTWISE_CASES = json.load(f)

case = LISTWISE_CASES[0]  # Greenhouse effect
print("🔧 LISTWISE RANKING (Technique #5)\n")
print(f"  Q: {case['question']}\n")
for a in case["answers"]:
    print(f"  [{a['label']}] {a['text'][:80]}...")

result = rank(case["question"], case["answers"])
if "parse_error" not in result:
    print(f"\n  LLM Ranking:      {result.get('ranking', [])}")
    print(f"  Expected Ranking: {case['expected_ranking']}")
else:
    print(f"\n  Parse error")

print("\n✅ CHECKPOINT: All 5 judging modes built + bias awareness complete")


---
## BLOCK 5: RELIABILITY & CONSISTENCY (70–85 min)
*Make your judge trustworthy with self-consistency, confidence thresholds, and component extraction.*

In [ ]:
# Cell 22 — Self-Consistency & Temperature Sensitivity (Techniques #21, #22, #27)

print("🔧 SELF-CONSISTENCY: Run N times, measure variance (Technique #21)\n")

case = TRICKY
scores = []
verdicts = []
for i in range(5):
    r = evaluate(case["question"], case["answer"])
    if "parse_error" not in r:
        s = r.get("overall_score", 0)
        scores.append(int(s) if str(s).isdigit() else 0)
        verdicts.append(r.get("verdict", "?"))

if scores:
    mean_s = statistics.mean(scores)
    std_s = statistics.stdev(scores) if len(scores) > 1 else 0
    print(f"  Scores across 5 runs: {scores}")
    print(f"  Mean: {mean_s:.1f}  Std Dev: {std_s:.2f}")
    print(f"  Verdicts: {verdicts}")
    verdict_counts = Counter(verdicts)
    majority = verdict_counts.most_common(1)[0]
    print(f"  Majority verdict: {majority[0]} ({majority[1]}/5 agreement)")

    # Cohen's Kappa approximation (Technique #27)
    # Simple agreement ratio between consecutive pairs
    agreements = sum(1 for i in range(len(verdicts)-1) if verdicts[i] == verdicts[i+1])
    agreement_ratio = agreements / (len(verdicts)-1) if len(verdicts) > 1 else 1
    print(f"\n  Inter-run agreement ratio: {agreement_ratio:.2f}")
    print(f"  {'✅ High consistency' if agreement_ratio >= 0.8 else '⚠️ Low consistency — consider ensemble'}")

print("\n🔧 TEMPERATURE SENSITIVITY (Technique #22)\n")
print("  Testing same eval at different temperatures:")
for temp in [0.0, 0.3, 0.7]:
    resp = completion(model=MODEL,
        messages=[{"role": "system", "content": EVALUATE_SYSTEM},
                  {"role": "user", "content": f"Question: {case['question']}\nAnswer: {case['answer']}"}],
        temperature=temp)
    raw = re.sub(r"```json\s*|\s*```", "", resp.choices[0].message.content.strip()).strip()
    try:
        parsed = json.loads(raw)
        print(f"  temp={temp}: score={parsed.get('overall_score','?')} verdict={parsed.get('verdict','?')}")
    except json.JSONDecodeError:
        print(f"  temp={temp}: parse error")

print("\n💡 LESSON: Variance is an uncertainty SIGNAL, not noise. High variance = ambiguous case.")


In [ ]:
# Cell 23 — Confidence Thresholds + Auto-Retry (Techniques #23, #24)

def evaluate_with_retry(question: str, answer: str, reference: str = None,
                        max_attempts: int = 3, min_confidence: str = "high") -> dict:
    """Evaluate with automatic retry on low confidence."""
    confidence_order = {"high": 3, "medium": 2, "low": 1}
    min_level = confidence_order.get(min_confidence, 2)
    all_attempts = []

    for attempt in range(max_attempts):
        result = evaluate(question, answer, reference)
        if "parse_error" in result:
            all_attempts.append({"attempt": attempt+1, "status": "parse_error"})
            continue

        conf = result.get("confidence", "low")
        conf_level = confidence_order.get(conf, 1)
        all_attempts.append({"attempt": attempt+1, "confidence": conf, "score": result.get("overall_score")})

        if conf_level >= min_level:
            return {"status": "accepted", "result": result, "attempts": attempt+1, "all_attempts": all_attempts}

    # All attempts below threshold → flag for human review
    best = max(all_attempts, key=lambda x: confidence_order.get(x.get("confidence","low"), 0))
    return {"status": "human_review", "reason": "Low confidence after all attempts",
            "attempts": len(all_attempts), "best_attempt": best, "all_attempts": all_attempts}

print("🔧 CONFIDENCE THRESHOLDS + AUTO-RETRY (Techniques #23, #24)\n")

# Test on easy case (should pass quickly)
r1 = evaluate_with_retry(EASY["question"], EASY["answer"], min_confidence="high")
print(f"  EASY case: status={r1['status']} attempts={r1['attempts']}")

# Test on adversarial case (may need retries)
r2 = evaluate_with_retry(ADVERSARIAL["question"], ADVERSARIAL["answer"], min_confidence="high")
print(f"  ADVERSARIAL case: status={r2['status']} attempts={r2['attempts']}")
if r2["status"] == "human_review":
    print(f"    → Flagged for human review: {r2['reason']}")

print("\n💡 LESSON: Automated quality gates prevent low-confidence evaluations from passing silently.")


In [ ]:
# Cell 24 — Component Extraction (Technique #25) + Claim Decomposition (Technique #26)

COMPONENT_SYSTEM = """You extract classification components. NEVER calculate metrics yourself.
Categorize each prediction by its INDEX.

Respond ONLY with valid JSON:
{
    "true_positives": [list of indices],
    "true_negatives": [list of indices],
    "false_positives": [list of indices],
    "false_negatives": [list of indices],
    "reasoning": "step-by-step categorization"
}"""

print("🔧 COMPONENT EXTRACTION: Never Let LLM Do Math (Technique #25)\n")

preds =  ["positive", "negative", "positive", "positive", "negative", "positive"]
truths = ["positive", "positive", "negative", "positive", "positive", "negative"]

indexed = "\n".join(f"[{i}] pred={p} truth={t}" for i, (p, t) in enumerate(zip(preds, truths)))
prompt = f"Categorize each prediction:\n{indexed}"
raw = ask_llm(prompt, system=COMPONENT_SYSTEM)
cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
try:
    components = json.loads(cleaned)
    tp = len(components.get("true_positives", []))
    tn = len(components.get("true_negatives", []))
    fp = len(components.get("false_positives", []))
    fn = len(components.get("false_negatives", []))
    accuracy = (tp + tn) / len(preds)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"  LLM extracted: TP={components.get('true_positives')}, FP={components.get('false_positives')}")
    print(f"  WE calculate:  Accuracy={accuracy:.3f}  Precision={precision:.3f}  Recall={recall:.3f}")
except json.JSONDecodeError:
    print(f"  Parse error: {raw[:200]}")

print("\n🔧 CLAIM DECOMPOSITION (Technique #26)\n")

CLAIM_SYSTEM = """Decompose the answer into individual atomic claims. Verify each claim independently.

Respond ONLY with valid JSON:
{
    "claims": [
        {"claim": "...", "verified": true, "evidence": "..."},
        {"claim": "...", "verified": false, "evidence": "..."}
    ],
    "total_claims": 0,
    "verified_count": 0,
    "factual_accuracy": "verified_count / total_claims"
}"""

case = TEST_CASES[11]  # Verbose exercise answer
prompt = f"Question: {case['question']}\nAnswer: {case['answer'][:300]}\n\nDecompose into atomic claims and verify each."
raw = ask_llm(prompt, system=CLAIM_SYSTEM)
cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
try:
    claims = json.loads(cleaned)
    total = claims.get("total_claims", len(claims.get("claims", [])))
    verified = claims.get("verified_count", sum(1 for c in claims.get("claims", []) if c.get("verified")))
    print(f"  Total claims: {total}  Verified: {verified}")
    for c in claims.get("claims", [])[:4]:
        status = "✅" if c.get("verified") else "❌"
        print(f"  {status} {c.get('claim','')[:80]}")
    if total > 4:
        print(f"  ... and {total - 4} more claims")
except json.JSONDecodeError:
    print(f"  Parse error: {raw[:200]}")

print("\n💡 LESSON: Extract components → compute metrics yourself. Decompose claims → verify individually.")


---
## BLOCK 6: MULTI-JUDGE ARCHITECTURES (85–100 min)
*One judge is fragile. Multiple judges are robust. Learn ensemble, specialized, meta-judge, cascading, and council patterns.*

In [ ]:
# Cell 25 — Ensemble Judges: Same-Model + Cross-Model (Techniques #28, #29, #30)

def ensemble_evaluate(question: str, answer: str, models: List[str] = None,
                      temperatures: List[float] = None) -> dict:
    """Run evaluation across multiple models or temperatures and aggregate."""
    models = models or [MODEL]
    temperatures = temperatures or [0.0, 0.3, 0.7]
    results = []

    for model in models:
        for temp in temperatures:
            try:
                resp = completion(model=model,
                    messages=[{"role": "system", "content": EVALUATE_SYSTEM},
                              {"role": "user", "content": f"Question: {question}\nAnswer: {answer}"}],
                    temperature=temp)
                raw = re.sub(r"```json\s*|\s*```", "", resp.choices[0].message.content.strip()).strip()
                parsed = json.loads(raw)
                results.append({"model": model, "temp": temp,
                                "score": int(parsed.get("overall_score", 0)),
                                "verdict": parsed.get("verdict", "?"),
                                "confidence": parsed.get("confidence", "?")})
            except Exception as e:
                results.append({"model": model, "temp": temp, "error": str(e)})

    # Aggregate
    valid = [r for r in results if "error" not in r]
    if not valid:
        return {"status": "all_failed", "results": results}

    scores = [r["score"] for r in valid]
    verdicts = [r["verdict"] for r in valid]
    verdict_counts = Counter(verdicts)

    return {
        "individual_results": valid,
        "mean_score": statistics.mean(scores),
        "std_score": statistics.stdev(scores) if len(scores) > 1 else 0,
        "majority_verdict": verdict_counts.most_common(1)[0][0],
        "agreement": verdict_counts.most_common(1)[0][1] / len(valid),
        "num_judges": len(valid)
    }

print("🔧 ENSEMBLE EVALUATION (Techniques #28, #29, #30)\n")
print("Same model, 3 temperatures on TRICKY case:\n")

result = ensemble_evaluate(TRICKY["question"], TRICKY["answer"])
print(f"  Judges: {result.get('num_judges', 0)}")
print(f"  Mean score: {result.get('mean_score', 0):.1f}  Std: {result.get('std_score', 0):.2f}")
print(f"  Majority verdict: {result.get('majority_verdict', '?')} ({result.get('agreement', 0):.0%} agreement)")
for r in result.get("individual_results", []):
    print(f"    {r['model']}@{r['temp']}: score={r['score']} verdict={r['verdict']}")

print("\n💡 LESSON: Multiple perspectives reduce noise. Disagreement = genuine uncertainty.")


In [ ]:
# Cell 26 — Specialized Judges + Meta-Judge (Techniques #31, #32)

ACCURACY_JUDGE = """You are an ACCURACY specialist. Evaluate ONLY factual correctness.
Respond ONLY with valid JSON:
{"accuracy_score": 0, "factual_errors": [], "verified_facts": [], "confidence": "high/medium/low"}"""

COMPLETENESS_JUDGE = """You are a COMPLETENESS specialist. Evaluate ONLY thoroughness and coverage.
Respond ONLY with valid JSON:
{"completeness_score": 0, "missing_elements": [], "covered_aspects": [], "confidence": "high/medium/low"}"""

CLARITY_JUDGE = """You are a CLARITY specialist. Evaluate ONLY how clear and well-structured the answer is.
Respond ONLY with valid JSON:
{"clarity_score": 0, "clarity_issues": [], "clear_elements": [], "confidence": "high/medium/low"}"""

META_JUDGE_SYSTEM = """You are a meta-judge. Analyze results from 3 specialized judges and produce a consensus.

Respond ONLY with valid JSON:
{
    "agreement_level": "high/medium/low",
    "consensus_verdict": "good/mixed/poor",
    "composite_score": 0,
    "disagreements": ["..."],
    "human_review_needed": true,
    "reasoning": "..."
}"""

def specialized_evaluation(question: str, answer: str) -> dict:
    """Run 3 specialized judges + meta-judge consensus."""
    prompt = f"Question: {question}\nAnswer: {answer}"
    judges = {"accuracy": ACCURACY_JUDGE, "completeness": COMPLETENESS_JUDGE, "clarity": CLARITY_JUDGE}
    results = {}

    for name, system in judges.items():
        raw = ask_llm(prompt, system=system)
        cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
        try:
            results[name] = json.loads(cleaned)
        except json.JSONDecodeError:
            results[name] = {"error": "parse_failed"}

    # Meta-judge
    meta_prompt = f"Analyze these judge results:\n{json.dumps(results, indent=2)}"
    meta_raw = ask_llm(meta_prompt, system=META_JUDGE_SYSTEM)
    cleaned = re.sub(r"```json\s*|\s*```", "", meta_raw).strip()
    try:
        meta = json.loads(cleaned)
    except json.JSONDecodeError:
        meta = {"error": "parse_failed", "raw": meta_raw[:200]}

    return {"judges": results, "meta": meta}

print("🔧 SPECIALIZED JUDGES + META-JUDGE (Techniques #31, #32)\n")
result = specialized_evaluation(TRICKY["question"], TRICKY["answer"])

for name, r in result["judges"].items():
    score_key = f"{name}_score"
    print(f"  {name.upper()} judge: score={r.get(score_key, '?')} conf={r.get('confidence','?')}")

meta = result["meta"]
if "error" not in meta:
    print(f"\n  META-JUDGE: consensus={meta.get('consensus_verdict','?')} "
          f"composite={meta.get('composite_score','?')} "
          f"human_review={meta.get('human_review_needed','?')}")
    if meta.get("disagreements"):
        print(f"  Disagreements: {meta['disagreements']}")

print("\n💡 LESSON: Specialization beats generalization. Meta-judge resolves disagreements.")


In [ ]:
# Cell 27 — Cascading Judges (Technique #33) + LLM Council Placeholder (Technique #34)

def cascading_evaluate(question: str, answer: str,
                       cheap_model: str = MODEL, expensive_model: str = MODEL,
                       confidence_threshold: str = "high") -> dict:
    """Cheap model evaluates first. If uncertain, escalate to expensive model."""
    confidence_order = {"high": 3, "medium": 2, "low": 1}

    # Stage 1: Cheap/fast model
    resp = completion(model=cheap_model,
        messages=[{"role": "system", "content": EVALUATE_SYSTEM},
                  {"role": "user", "content": f"Question: {question}\nAnswer: {answer}"}],
        temperature=0)
    raw = re.sub(r"```json\s*|\s*```", "", resp.choices[0].message.content.strip()).strip()
    try:
        stage1 = json.loads(raw)
    except json.JSONDecodeError:
        stage1 = {"confidence": "low", "overall_score": "?"}

    conf = confidence_order.get(stage1.get("confidence", "low"), 1)
    threshold = confidence_order.get(confidence_threshold, 3)

    if conf >= threshold:
        return {"resolved_by": "cheap_model", "result": stage1, "escalated": False}

    # Stage 2: Expensive/powerful model
    resp2 = completion(model=expensive_model,
        messages=[{"role": "system", "content": EVALUATE_SYSTEM},
                  {"role": "user", "content": f"Question: {question}\nAnswer: {answer}"}],
        temperature=0)
    raw2 = re.sub(r"```json\s*|\s*```", "", resp2.choices[0].message.content.strip()).strip()
    try:
        stage2 = json.loads(raw2)
    except json.JSONDecodeError:
        stage2 = {"confidence": "low", "overall_score": "?"}

    return {"resolved_by": "expensive_model", "result": stage2, "escalated": True,
            "cheap_result": stage1}

print("🔧 CASCADING JUDGES (Technique #33)\n")

results = []
for i, case in enumerate(TEST_CASES[:6]):
    r = cascading_evaluate(case["question"], case["answer"])
    results.append(r)
    esc = "⬆️ ESCALATED" if r["escalated"] else "✅ resolved"
    print(f"  Case {i+1}: {esc} by {r['resolved_by']} — score={r['result'].get('overall_score','?')}")

escalated = sum(1 for r in results if r["escalated"])
print(f"\n  {len(results) - escalated}/{len(results)} resolved by cheap model, {escalated} escalated")

print("\n" + "=" * 60)
print("🔧 LLM COUNCIL — Multi-Round Deliberation (Technique #34)")
print("=" * 60)
print("""
  The LLM Council pattern involves multi-round deliberation:
    Round 1: Multiple models evaluate independently
    Round 2: Each model sees others' reasoning, can update position
    Round 3: Meta-judge synthesizes final verdict

  This is a PLACEHOLDER — the full implementation will be added
  with a dedicated GitHub repository for production use.

  For now, the ensemble + meta-judge pattern (Techniques #30, #32)
  provides a simpler version of this approach.
""")
print("✅ CHECKPOINT: Complete multi-judge architecture built")


---
## BLOCK 7: ADVANCED TECHNIQUES (100–112 min)
*Dynamic rubrics, hallucination detection, domain adaptation, multi-turn evaluation, and more.*

In [ ]:
# Cell 28 — Dynamic Rubric Generation + Calibrated Anchors + Domain Adaptation (Techniques #35, #36, #37)

RUBRIC_GEN_SYSTEM = """You generate evaluation rubrics for specific tasks. Given a task description,
create a scoring rubric with clear criteria for scores 1, 3, 5, 7, and 9.

Respond ONLY with valid JSON:
{
    "rubric": {
        "1": "description of score 1",
        "3": "description of score 3",
        "5": "description of score 5",
        "7": "description of score 7",
        "9": "description of score 9"
    },
    "key_dimensions": ["dim1", "dim2", "dim3"]
}"""

print("🔧 DYNAMIC RUBRIC GENERATION (Technique #35)\n")
task_desc = "Evaluate medical advice answers for patient safety and accuracy"
raw = ask_llm(f"Generate an evaluation rubric for: {task_desc}", system=RUBRIC_GEN_SYSTEM)
cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
try:
    rubric = json.loads(cleaned)
    print(f"  Generated rubric for: '{task_desc}'")
    for score, desc in rubric.get("rubric", {}).items():
        print(f"    {score}/9: {desc[:80]}")
except json.JSONDecodeError:
    print(f"  Parse error: {raw[:200]}")

print("\n🔧 CALIBRATED SCORING WITH ANCHORS (Technique #36)\n")
ANCHOR_SYSTEM = """You evaluate answers using calibrated anchor examples.

ANCHOR EXAMPLES (use these to calibrate your scoring):
- Score 3/9: "Rain happens when clouds get heavy." (oversimplified, missing mechanism)
- Score 6/9: "Rain forms from water vapor condensing in clouds." (correct but basic)
- Score 9/9: "Rain forms when water vapor condenses into droplets in clouds. When heavy enough to overcome air resistance, they fall as precipitation. Driven by humidity, temperature changes, and pressure systems." (comprehensive)

Score the following answer relative to these anchors.
Respond ONLY with valid JSON: {"calibrated_score": 0, "closest_anchor": "3/6/9", "justification": "..."}"""

anchor_result = ask_llm(f"Question: {TRICKY['question']}\nAnswer: {TRICKY['answer']}", system=ANCHOR_SYSTEM)
cleaned = re.sub(r"```json\s*|\s*```", "", anchor_result).strip()
try:
    parsed = json.loads(cleaned)
    print(f"  Calibrated score: {parsed.get('calibrated_score')}/9 (closest anchor: {parsed.get('closest_anchor')})")
except json.JSONDecodeError:
    print(f"  Parse error")

print("\n🔧 DOMAIN-ADAPTED JUDGES (Technique #37)\n")
medical_case = TEST_CASES[6]  # Heart attack warning signs
general_result = evaluate(medical_case["question"], medical_case["answer"])
# Domain-specific prompt
domain_prompt = f"You are a MEDICAL SAFETY evaluator. Patient lives depend on completeness.\nQuestion: {medical_case['question']}\nAnswer: {medical_case['answer']}"
domain_result_raw = ask_llm(domain_prompt, system=EVALUATE_SYSTEM)
cleaned = re.sub(r"```json\s*|\s*```", "", domain_result_raw).strip()
try:
    domain_result = json.loads(cleaned)
    print(f"  General judge:   score={general_result.get('overall_score','?')}")
    print(f"  Medical judge:   score={domain_result.get('overall_score','?')}")
    print(f"  (Medical judge should be stricter on completeness for safety-critical info)")
except json.JSONDecodeError:
    print(f"  Parse error")

print("\n💡 LESSON: Generate rubrics dynamically, anchor scores with examples, adapt to domains.")


In [ ]:
# Cell 29 — Multi-Turn Conversation Eval + Hallucination Detection (Techniques #38, #39)

CONVERSATION_JUDGE = """You evaluate multi-turn conversations. Assess EACH turn and the overall flow.

Respond ONLY with valid JSON:
{
    "turn_scores": [{"turn": 1, "score": 0, "issue": "..."}],
    "coherence": 0,
    "helpfulness": 0,
    "escalation_handling": 0,
    "overall_score": 0,
    "worst_turn": 1,
    "recommendation": "..."
}"""

with open("../datasets/multi_turn_cases.json") as f:
    MULTI_TURN_CASES = json.load(f)

print("🔧 MULTI-TURN CONVERSATION EVALUATION (Technique #38)\n")
conv = MULTI_TURN_CASES[1]  # Mediocre tech support
conv_text = "\n".join(f"  {m['role'].upper()}: {m['content'][:80]}..." for m in conv["conversation"])
print(f"  Scenario: {conv['scenario']}\n{conv_text}\n")

prompt = f"Evaluate this conversation:\n{json.dumps(conv['conversation'], indent=2)}"
raw = ask_llm(prompt, system=CONVERSATION_JUDGE)
cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
try:
    result = json.loads(cleaned)
    print(f"  Overall: {result.get('overall_score','?')}/9  Worst turn: {result.get('worst_turn','?')}")
    print(f"  Human baseline: coherence={conv['quality_scores']['coherence']} helpfulness={conv['quality_scores']['helpfulness']}")
except json.JSONDecodeError:
    print(f"  Parse error: {raw[:200]}")

print("\n🔧 HALLUCINATION DETECTION PIPELINE (Technique #39)\n")

HALLUCINATION_SYSTEM = """You detect hallucinations by comparing an answer against source documents.
A hallucination is any claim NOT supported by the provided source documents, even if factually true.

Respond ONLY with valid JSON:
{
    "claims_analyzed": [
        {"claim": "...", "in_source": true, "evidence": "..."}
    ],
    "hallucinated_claims": ["..."],
    "supported_claims": ["..."],
    "hallucination_rate": "X/Y claims unsupported"
}"""

with open("../datasets/hallucination_cases.json") as f:
    HALLUCINATION_CASES = json.load(f)

hcase = HALLUCINATION_CASES[0]  # Treaty of Versailles
prompt = (f"Answer: {hcase['answer'][:400]}\n\n"
          f"Source Documents: {hcase['source_documents']}\n\n"
          f"Identify any claims NOT supported by the source documents.")
raw = ask_llm(prompt, system=HALLUCINATION_SYSTEM)
cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
try:
    result = json.loads(cleaned)
    print(f"  Hallucination rate: {result.get('hallucination_rate', '?')}")
    for h in result.get("hallucinated_claims", [])[:3]:
        print(f"    ❌ {h[:80]}")
    print(f"  Known hallucinations: {hcase['hallucinated_claims'][0][:80]}...")
except json.JSONDecodeError:
    print(f"  Parse error: {raw[:200]}")

print("\n💡 LESSON: Conversation eval needs turn-aware prompts. Hallucination detection is claim-level.")


In [ ]:
# Cell 30 — Logprob-Based Confidence + When to Fine-Tune (Techniques #40, #41)

print("🔧 LOGPROB-BASED CONFIDENCE (Technique #40)\n")
print("""  When your API supports logprobs (OpenAI, some Ollama models), you can extract
  token-level probabilities as a confidence signal:

  Example (pseudo-code):
  ┌──────────────────────────────────────────────────────────────────┐
  │  resp = completion(model="gpt-4o-mini", messages=[...],         │
  │                    logprobs=True, top_logprobs=5)                │
  │                                                                  │
  │  # Extract probability of "Yes" vs "No" token                   │
  │  token_probs = resp.choices[0].logprobs.content[0].top_logprobs │
  │  for tp in token_probs:                                         │
  │      print(f"  Token: {tp.token}  Prob: {exp(tp.logprob):.3f}") │
  │                                                                  │
  │  # Output:                                                       │
  │  #   Token: Yes  Prob: 0.847                                     │
  │  #   Token: No   Prob: 0.142                                     │
  │  #   Token: Maybe Prob: 0.011                                    │
  └──────────────────────────────────────────────────────────────────┘

  When available, logprobs are the most DIRECT confidence measure —
  no need to ask the LLM "how confident are you?" (which it often gets wrong).
""")

print("🔧 WHEN TO FINE-TUNE A JUDGE (Technique #41)\n")
print("""  Decision framework — try these IN ORDER before fine-tuning:

  1. Better prompts (structured, CoT, few-shot)     ← usually enough
  2. Better rubrics (domain-specific, calibrated)    ← often enough
  3. Ensemble of multiple models/prompts             ← usually sufficient
  4. Fine-tune a judge model                         ← nuclear option

  When fine-tuning IS justified:
  • You have 1000+ human-labeled evaluation examples
  • Prompt engineering plateaus below acceptable accuracy
  • You need sub-second latency (smaller fine-tuned model)
  • Domain is highly specialized (legal, medical, code review)

  Resources for judge fine-tuning:
  • Prometheus (judge model fine-tuning framework)
  • JudgeLM (training LLMs as judges)
  • Auto-J (automated judge training pipeline)
  • MT-Bench (evaluation benchmark for judge models)
""")


---
## BLOCK 8: PRODUCTION & BENCHMARKING (112–118 min)
*Benchmark your judge against humans. Build production monitoring patterns.*

In [ ]:
# Cell 31 — Judge Benchmarking Against Human Labels (Technique #42)

with open("../datasets/human_baseline.json") as f:
    HUMAN_BASELINE = json.load(f)

print("🔧 JUDGE BENCHMARKING AGAINST HUMANS (Technique #42)\n")
print("Running evaluate() on all 12 test cases and comparing to human expert scores...\n")

human_scores = {a["case_id"]: a for a in HUMAN_BASELINE["annotations"]}
judge_results = []

for case in TEST_CASES:
    result = evaluate(case["question"], case["answer"], reference=case.get("reference_answer"))
    human = human_scores.get(case["id"], {})

    judge_score = result.get("overall_score", 0) if "parse_error" not in result else "?"
    judge_verdict = result.get("verdict", "?") if "parse_error" not in result else "?"
    human_score = human.get("overall", "?")
    human_verdict = human.get("verdict", "?")

    match = "✅" if judge_verdict == human_verdict else "❌"
    judge_results.append({
        "case_id": case["id"], "judge_score": judge_score, "human_score": human_score,
        "judge_verdict": judge_verdict, "human_verdict": human_verdict, "match": judge_verdict == human_verdict
    })
    print(f"  Case {case['id']:>2}: Judge={judge_score}/9 ({judge_verdict:<5}) "
          f"Human={human_score}/9 ({human_verdict:<5}) {match}")

# Calculate agreement metrics
valid = [r for r in judge_results if r["judge_score"] != "?" and r["human_score"] != "?"]
if valid:
    agreement = sum(1 for r in valid if r["match"]) / len(valid)
    score_diffs = [abs(int(r["judge_score"]) - int(r["human_score"])) for r in valid
                   if str(r["judge_score"]).isdigit() and str(r["human_score"]).isdigit()]
    avg_diff = np.mean(score_diffs) if score_diffs else 0
    correlation = np.corrcoef(
        [int(r["judge_score"]) for r in valid if str(r["judge_score"]).isdigit()],
        [int(r["human_score"]) for r in valid if str(r["human_score"]).isdigit()]
    )[0, 1] if len(score_diffs) > 2 else 0

    print(f"\n📊 BENCHMARKING RESULTS:")
    print(f"  Verdict agreement: {agreement:.0%} ({sum(r['match'] for r in valid)}/{len(valid)})")
    print(f"  Avg score difference: {avg_diff:.1f} points")
    print(f"  Score correlation: {correlation:.2f}")

print("\n💡 LESSON: ALWAYS benchmark against humans before deploying. Numbers don't lie.")


In [ ]:
# Cell 32 — Production Patterns (Techniques #43-46)

print("🔧 PRODUCTION PATTERNS (Techniques #43-46)\n")

# Technique #43 — Score Drift Monitoring
print("━━━ SCORE DRIFT MONITORING (#43) ━━━")
print("""
  Track score distributions over time. Alert when they shift.

  Template:
  ┌──────────────────────────────────────────────────────────────┐
  │  import datetime                                             │
  │                                                              │
  │  def log_evaluation(case_id, score, verdict, confidence):    │
  │      entry = {                                               │
  │          "timestamp": datetime.datetime.utcnow().isoformat(),│
  │          "case_id": case_id,                                 │
  │          "score": score,                                     │
  │          "verdict": verdict,                                 │
  │          "confidence": confidence,                           │
  │          "model": MODEL                                      │
  │      }                                                       │
  │      # Append to log file / database / monitoring system     │
  │      return entry                                            │
  │                                                              │
  │  def check_drift(recent_scores, baseline_mean, threshold=1): │
  │      current_mean = np.mean(recent_scores)                   │
  │      drift = abs(current_mean - baseline_mean)               │
  │      if drift > threshold:                                   │
  │          alert(f"Score drift: {drift:.2f} from baseline")    │
  └──────────────────────────────────────────────────────────────┘
""")

# Technique #44 — Cost Optimization
print("━━━ COST OPTIMIZATION (#44) ━━━")
print("""
  • Batch evaluations: Group API calls, reduce overhead
  • Cache identical inputs: Hash (question + answer) → return cached result
  • Model routing: Easy cases → cheap model, hard cases → expensive model
  • Async evaluation: Run multiple judges concurrently with asyncio
""")

# Technique #45 — Audit Trail
print("━━━ AUDIT TRAIL & EXPLAINABILITY (#45) ━━━")

def evaluate_with_audit(question: str, answer: str) -> dict:
    """Evaluate and log complete audit trail."""
    import datetime
    result = evaluate(question, answer)
    audit = {
        "timestamp": datetime.datetime.utcnow().isoformat(),
        "model": MODEL,
        "input": {"question": question, "answer": answer[:200]},
        "output": result,
        "system_prompt_hash": hash(EVALUATE_SYSTEM),
    }
    return audit

sample_audit = evaluate_with_audit(EASY["question"], EASY["answer"])
print(f"  Audit entry keys: {list(sample_audit.keys())}")
print(f"  Timestamp: {sample_audit['timestamp']}")
print(f"  Verdict: {sample_audit['output'].get('verdict', '?')}")

# Technique #46 — Human Fallback
print("\n━━━ HUMAN-IN-THE-LOOP FALLBACK (#46) ━━━")
print("""
  Automatic escalation triggers:
  • Confidence < threshold         → human review
  • Judge disagreement > threshold → human review
  • Score in ambiguous range       → human review
  • Flagged domain (medical/legal) → mandatory human review

  Integration pattern:
    if result["confidence"] == "low" or result.get("human_review_needed"):
        send_to_human_queue(case_id, result, priority="high")
    else:
        auto_approve(case_id, result)
""")

print("💡 LESSON: Production judges need monitoring, caching, audit trails, and human fallbacks.")


---
## BLOCK 9: WRAP-UP (118–120 min)
*Your complete `LLMJudge` class, cheat sheet, and what to do next.*

In [ ]:
# Cell 33 — Complete LLMJudge Class: Everything you built, in one reusable package

class LLMJudge:
    """Production-grade LLM-as-a-Judge combining all 46 techniques."""

    def __init__(self, model: str = MODEL, cheap_model: str = None):
        self.model = model
        self.cheap_model = cheap_model or model

    def _ask(self, prompt: str, system: str, model: str = None, temperature: float = 0) -> str:
        resp = completion(model=model or self.model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
            temperature=temperature)
        return resp.choices[0].message.content.strip()

    def _parse_json(self, raw: str) -> dict:
        cleaned = re.sub(r"```json\s*|\s*```", "", raw).strip()
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return {"parse_error": True, "raw": raw}

    # ── Core Evaluation (Techniques #6-12) ────────────────────────
    def evaluate(self, question: str, answer: str, reference: str = None) -> dict:
        """Structured + rubric + CoT + oath evaluation."""
        prompt = f"Question: {question}\nAnswer: {answer}"
        if reference:
            prompt += f"\nReference Answer: {reference}\nCompare candidate against reference."
        return self._parse_json(self._ask(prompt, EVALUATE_SYSTEM))

    # ── Adversarial Critique (Technique #13) ──────────────────────
    def critique(self, question: str, answer: str) -> dict:
        """Find all flaws first, then mention strengths."""
        prompt = f"Question: {question}\nAnswer: {answer}\n\nFind ALL problems."
        return self._parse_json(self._ask(prompt, CRITIQUE_SYSTEM))

    # ── Pairwise Comparison (Techniques #4, #15, #16) ─────────────
    def compare(self, question: str, answer_a: str, answer_b: str, debias: bool = True) -> dict:
        """Pairwise comparison with optional position debiasing."""
        prompt = f"Question: {question}\nAnswer A: {answer_a}\nAnswer B: {answer_b}"
        r1 = self._parse_json(self._ask(prompt, PAIRWISE_SYSTEM))
        if not debias:
            return r1
        # Swap and compare again
        prompt2 = f"Question: {question}\nAnswer A: {answer_b}\nAnswer B: {answer_a}"
        r2 = self._parse_json(self._ask(prompt2, PAIRWISE_SYSTEM))
        w1 = r1.get("winner", "?")
        w2_raw = r2.get("winner", "?")
        w2 = {"A": "B", "B": "A", "tie": "tie"}.get(w2_raw, w2_raw)
        if w1 == w2:
            return {"winner": w1, "consistent": True, "details": [r1, r2]}
        return {"winner": "tie", "consistent": False, "note": f"Inconsistent: {w1} vs {w2}"}

    # ── Listwise Ranking (Technique #5) ───────────────────────────
    def rank(self, question: str, answers: list) -> dict:
        answers_text = "\n".join(f"Answer {a['label']}: {a['text']}" for a in answers)
        prompt = f"Question: {question}\n\n{answers_text}\n\nRank from best to worst."
        return self._parse_json(self._ask(prompt, RANKING_SYSTEM))

    # ── Robust Evaluation (Techniques #21, #24) ───────────────────
    def evaluate_robust(self, question: str, answer: str, reference: str = None,
                        runs: int = 3, min_confidence: str = "medium") -> dict:
        """Multi-run evaluation with confidence thresholds."""
        confidence_order = {"high": 3, "medium": 2, "low": 1}
        min_level = confidence_order.get(min_confidence, 2)
        results = []
        for _ in range(runs):
            r = self.evaluate(question, answer, reference)
            if "parse_error" not in r:
                results.append(r)
        if not results:
            return {"status": "all_failed"}
        scores = [int(r.get("overall_score", 0)) for r in results if str(r.get("overall_score", "")).isdigit()]
        verdicts = [r.get("verdict", "?") for r in results]
        verdict_counts = Counter(verdicts)
        best_conf = max(results, key=lambda x: confidence_order.get(x.get("confidence", "low"), 0))
        return {
            "status": "accepted" if confidence_order.get(best_conf.get("confidence"), 0) >= min_level else "human_review",
            "mean_score": statistics.mean(scores) if scores else 0,
            "std_score": statistics.stdev(scores) if len(scores) > 1 else 0,
            "majority_verdict": verdict_counts.most_common(1)[0][0],
            "agreement": verdict_counts.most_common(1)[0][1] / len(results),
            "runs": len(results)
        }

    # ── Specialized + Meta-Judge (Techniques #31, #32) ────────────
    def evaluate_specialized(self, question: str, answer: str) -> dict:
        """3 specialized judges + meta-judge consensus."""
        prompt = f"Question: {question}\nAnswer: {answer}"
        judges = {"accuracy": ACCURACY_JUDGE, "completeness": COMPLETENESS_JUDGE, "clarity": CLARITY_JUDGE}
        results = {}
        for name, sys in judges.items():
            results[name] = self._parse_json(self._ask(prompt, sys))
        meta_prompt = f"Analyze these judge results:\n{json.dumps(results, indent=2)}"
        meta = self._parse_json(self._ask(meta_prompt, META_JUDGE_SYSTEM))
        return {"judges": results, "meta": meta}

    # ── Cascading (Technique #33) ─────────────────────────────────
    def evaluate_cascading(self, question: str, answer: str) -> dict:
        """Cheap model first, escalate to expensive if uncertain."""
        prompt = f"Question: {question}\nAnswer: {answer}"
        r1 = self._parse_json(self._ask(prompt, EVALUATE_SYSTEM, model=self.cheap_model))
        if r1.get("confidence") == "high" and "parse_error" not in r1:
            return {"resolved_by": "cheap", "result": r1, "escalated": False}
        r2 = self._parse_json(self._ask(prompt, EVALUATE_SYSTEM, model=self.model))
        return {"resolved_by": "expensive", "result": r2, "escalated": True, "cheap_result": r1}

    # ── Full Pipeline ─────────────────────────────────────────────
    def full_pipeline(self, question: str, answer: str, reference: str = None) -> dict:
        """Run the complete evaluation pipeline."""
        eval_result = self.evaluate(question, answer, reference)
        crit_result = self.critique(question, answer)
        return {
            "evaluation": eval_result,
            "critique": crit_result,
            "score": eval_result.get("overall_score", "?"),
            "verdict": eval_result.get("verdict", "?"),
            "flaws": crit_result.get("critical_flaws", []),
            "confidence": eval_result.get("confidence", "?"),
            "human_review": eval_result.get("confidence") == "low"
        }

# Demo
judge = LLMJudge(model=MODEL)
print("✅ LLMJudge class built! Methods available:")
for method in ["evaluate", "critique", "compare", "rank", "evaluate_robust",
               "evaluate_specialized", "evaluate_cascading", "full_pipeline"]:
    print(f"  • judge.{method}()")

print("\nQuick test — full_pipeline on TRICKY case:")
result = judge.full_pipeline(TRICKY["question"], TRICKY["answer"], TRICKY["reference_answer"])
print(f"  Score: {result['score']}/9  Verdict: {result['verdict']}  Confidence: {result['confidence']}")
print(f"  Flaws: {result['flaws'][:2]}")
print(f"  Human review needed: {result['human_review']}")


In [ ]:
# Cell 34 — Cheat Sheet: All 46 Techniques at a Glance

CHEAT_SHEET = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║                    LLM-AS-A-JUDGE TECHNIQUE CHEAT SHEET                        ║
╚══════════════════════════════════════════════════════════════════════════════════╝

FOUNDATIONAL MODES
  #1  Binary Yes/No           Simplest — use only for clear-cut checks
  #2  Likert Scale (1-10)     Quick rating — needs rubric to be useful
  #3  Pointwise Scoring       Single-answer score — backbone of most systems
  #4  Pairwise Comparison     A vs B — clearer signal than absolute scores
  #5  Listwise Ranking        Rank N answers — best for selection tasks

PROMPT ENGINEERING
  #6  Structured JSON Output  ★ MOST IMPORTANT — makes everything parseable
  #7  Explicit Rubric         Define exactly what each score means
  #8  Decomposed Criteria     Break "good" into measurable dimensions
  #9  Chain-of-Thought        Reason BEFORE scoring — catches more issues
  #10 Oath-Based Consistency  Behavioral commitment reduces scoring drift
  #11 Few-Shot Examples       Anchor scoring scale with example evaluations
  #12 Reference-Grounded      Compare against gold answer, not internal knowledge
  #13 Adversarial Critique    Find ALL flaws FIRST — counters LLM generosity
  #14 Contrastive Evaluation  "What SPECIFICALLY makes A better than B?"

BIAS DETECTION & MITIGATION
  #15 Position Bias Detection  Run A-B and B-A, check consistency
  #16 Position Debiasing       Swap orders + take consistent winner
  #17 Length Bias Detection     Short correct vs long wrong — does LLM prefer long?
  #18 Length Debiasing          "Evaluate SUBSTANCE not LENGTH" in prompt
  #19 Sycophancy Detection     Confident-but-wrong → does LLM agree?
  #20 Anchor Bias Detection    Prior score in context → does it inflate?

RELIABILITY & CONSISTENCY
  #21 Self-Consistency         Run N times, majority vote — variance = uncertainty
  #22 Temperature Sensitivity  Same eval at different temps — fragile if variable
  #23 Confidence Calibration   Does "high confidence" = actually correct?
  #24 Confidence Thresholds    Auto-retry low-confidence, flag for human review
  #25 Component Extraction     NEVER let LLM do math — extract, compute yourself
  #26 Claim Decomposition      Break into atomic claims → verify each
  #27 Inter-Run Agreement      Cohen's Kappa — quantify reliability

MULTI-JUDGE ARCHITECTURES
  #28 Same-Model (Diff Prompts)  Prompt diversity → perspective diversity
  #29 Same-Model (Diff Temps)    Cheap ensemble from one model
  #30 Cross-Model Ensemble       Different LLMs = different blind spots
  #31 Specialized Judges         Accuracy + Completeness + Clarity specialists
  #32 Meta-Judge                 Judge-of-judges resolves disagreements
  #33 Cascading Judges           Cheap first → expensive if uncertain
  #34 LLM Council                Multi-round deliberation for high stakes

ADVANCED TECHNIQUES
  #35 Dynamic Rubric Generation  LLM writes rubric → uses it to evaluate
  #36 Calibrated Anchor Scoring  Anchor examples define the score scale
  #37 Domain-Adapted Judges      Swap rubric + expertise for each domain
  #38 Multi-Turn Conversation    Turn-aware prompts for dialogue eval
  #39 Hallucination Detection    Claim decomposition + source cross-reference
  #40 Logprob Confidence         Token probabilities = direct confidence
  #41 Fine-Tuning Judges         Nuclear option when prompting isn't enough

PRODUCTION & OPERATIONS
  #42 Human Benchmarking         Compare judge scores to human experts
  #43 Score Drift Monitoring     Track distributions, alert on shifts
  #44 Cost Optimization          Cache, batch, route by difficulty
  #45 Audit Trail                Log every evaluation with full context
  #46 Human-in-the-Loop          Auto-escalate uncertain cases

DECISION FLOWCHART:
  Quick check        → Single structured judge (#6-9)
  Important decision → Multi-run + confidence threshold (#21, #24)
  High-stakes        → Ensemble + adversarial + meta-judge (#13, #30, #32)
  Critical/regulated → LLM Council + human-in-the-loop (#34, #46)
"""

print(CHEAT_SHEET)


---
### What's Next

**You built 46 techniques from scratch in 2 hours. This notebook is your reference. Go build.**

#### Key Papers
- [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685) — Zheng et al., 2023
- [Prometheus: Inducing Fine-Grained Evaluation Capability](https://arxiv.org/abs/2310.08491) — Kim et al., 2023
- [JudgeLM: Fine-tuned Large Language Models are Scalable Judges](https://arxiv.org/abs/2310.17631) — Zhu et al., 2023
- [Auto-J: Generative Judge for Evaluating Alignment](https://arxiv.org/abs/2310.05470) — Li et al., 2023

#### LLM Council (Bonus — Coming Soon)
*A full multi-round deliberation system with dedicated GitHub repository will be added here.*

#### Connect
- Workshop repo: *(link will be added after push)*
- LLM Council repo: *(placeholder — coming soon)*